# Avance 3 — Modelo Baseline con S3 Vectors
_____

_Proyecto Integrador MNA 10_

**Equipo 16**:
- Joel
- Roberto
- David

**Fecha**: 24 de Mayo, 2026


# Objetivos

* 3.1 Establecer las medidas de calidad del modelo de aprendizaje automático.
* 3.2 Proporcionar un marco de referencia para evaluar y mejorar modelos más avanzados.

## Contexto

En el Avance 2 implementamos un pipeline de ingesta local con **ChromaDB** como vector store y el modelo de embeddings `paraphrase-multilingual-MiniLM-L12-v2` (384 dimensiones, ejecución local). Ese enfoque sirvió como prueba de concepto para validar la viabilidad del sistema RAG.

En este avance migramos a **Amazon S3 Vectors** como vector store en la nube, combinado con **Amazon Titan Text Embeddings V2** (1024 dimensiones) como modelo de embeddings. Esta migración permite:

1. **Escalabilidad**: S3 Vectors escala automáticamente sin gestión de infraestructura.
2. **Durabilidad**: Los vectores heredan la durabilidad de 11 nueves de S3.
3. **Integración nativa**: Compatible con Bedrock Knowledge Bases para RAG gestionado.

### Preguntas del avance

| Pregunta | Respuesta en nuestro contexto |
|---|---|
| ¿Qué algoritmo como baseline? | Búsqueda semántica por similitud coseno sobre embeddings de Titan V2 (1024d) |
| ¿Importancia de características? | Las dimensiones del embedding capturan semántica; el chunking y metadata (source, folder) son las features clave |
| ¿Sub/sobreajuste? | No aplica en sentido clásico — evaluamos calidad de retrieval (relevancia de chunks recuperados) |
| ¿Métrica adecuada? | Similitud coseno (0-1) como proxy de relevancia semántica; score ≥ 0.75 = alta relevancia |
| ¿Desempeño mínimo? | El baseline debe recuperar chunks relevantes con score > 0.5 para queries de dominio ISO 27001 |


### 1. Configuración del Entorno

En esta sección configuramos los clientes de AWS necesarios: **S3 Vectors** para almacenamiento vectorial y **Bedrock Runtime** para generación de embeddings.


In [1]:
# Librerias de Python
import json
import sys
import urllib3
from pathlib import Path

import boto3
from botocore.exceptions import ClientError

In [2]:
# Suprimir advertencias SSL (configuración de red local)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Configuración de AWS
AWS_REGION = "us-west-2"
AWS_PROFILE = "Developer"

session = boto3.Session(profile_name=AWS_PROFILE, region_name=AWS_REGION)
s3vectors = session.client("s3vectors", verify=False)
bedrock = session.client("bedrock-runtime", verify=False)
s3_client = session.client("s3", verify=False)

# Constantes del proyecto
VECTOR_BUCKET_NAME = "iso27001-vectors-e16"
INDEX_NAME = "iso27001-compliance-index"
S3_DATA_BUCKET = "mna-s3-proyecto-integrador"
EMBEDDING_MODEL_ID = "amazon.titan-embed-text-v2:0"
EMBEDDING_DIMENSIONS = 1024
CHUNK_SIZE = 500
CHUNK_OVERLAP = 80

print(f"Region: {AWS_REGION}")
print(f"Vector Bucket: {VECTOR_BUCKET_NAME}")
print(f"Index: {INDEX_NAME}")
print(f"Embedding Model: {EMBEDDING_MODEL_ID} ({EMBEDDING_DIMENSIONS}d)")


Region: us-west-2
Vector Bucket: iso27001-vectors-e16
Index: iso27001-compliance-index
Embedding Model: amazon.titan-embed-text-v2:0 (1024d)


### 2. Creación del Vector Bucket en S3

Un **vector bucket** es un tipo especial de bucket de S3 diseñado para almacenar y consultar datos vectoriales. A diferencia de un bucket general, usa APIs dedicadas optimizadas para operaciones vectoriales.

> **Nota**: El nombre del vector bucket debe ser único dentro de la región AWS.


In [3]:
# Crear el vector bucket (idempotente — maneja ConflictException si ya existe)
try:
    response = s3vectors.create_vector_bucket(vectorBucketName=VECTOR_BUCKET_NAME)
    print(f"[INFO] Vector bucket created: {response['vectorBucketArn']}")
except ClientError as e:
    if e.response['Error']['Code'] == 'ConflictException':
        print(f"[INFO] Vector bucket '{VECTOR_BUCKET_NAME}' already exists — reusing.")
    else:
        raise


[INFO] Vector bucket created: arn:aws:s3vectors:us-west-2:645624302113:bucket/iso27001-vectors-e16


### 3. Creación del Vector Index

El **vector index** define cómo se almacenan y buscan los vectores dentro del bucket. Configuramos:

- **Dimensión**: 1024 (salida de Titan Text Embeddings V2)
- **Métrica de distancia**: Coseno (mide similitud angular — estándar para búsqueda semántica)
- **Metadata no filtrable**: `source_text` (almacena el texto original del chunk sin indexarlo para filtrado)

| Parámetro | Valor | Justificación |
|---|---|---|
| Dimensiones | 1024 | Recomendación de Amazon Titan Embed V2 |
| Métrica | Coseno | Estándar para similitud semántica en NLP |
| Metadata filtrable | `source`, `folder`, `chunk` | Permite filtrar resultados por origen |
| Metadata no filtrable | `source_text` | Almacena texto sin costo de indexación |


In [6]:
# Crear el índice vectorial (idempotente)
try:
    response = s3vectors.create_index(
        vectorBucketName=VECTOR_BUCKET_NAME,
        indexName=INDEX_NAME,
        dimension=EMBEDDING_DIMENSIONS,
        distanceMetric="cosine",
        dataType="float32",
        metadataConfiguration={
            "nonFilterableMetadataKeys": ["source_text"]
        }
    )
    print(f"[INFO] Vector index created: {INDEX_NAME}")
    print(f"       Dimension: {EMBEDDING_DIMENSIONS}, Metric: cosine")
except ClientError as e:
    if e.response['Error']['Code'] == 'ConflictException':
        print(f"[INFO] Vector index '{INDEX_NAME}' already exists — reusing.")
    else:
        raise


[INFO] Vector index created: iso27001-compliance-index
       Dimension: 1024, Metric: cosine


### 4. Extracción de Texto y Chunking

Reutilizamos las funciones de extracción del Avance 2 (`extract_pdf`, `extract_docx`, `chunk_text`) para procesar los documentos de la base de conocimiento. El pipeline es idéntico:

1. **Extracción**: PDF → `pdfplumber`, DOCX → `python-docx`
2. **Chunking**: Fragmentos de ~500 caracteres con solapamiento de ~80 caracteres

La diferencia clave es que los embeddings ahora se generan con **Titan V2** (1024d) en lugar del modelo local MiniLM (384d).


In [ ]:
# Importar funciones de extracción del módulo de ingesta existente
sys.path.insert(0, '../src')
from rag.ingest import extract_pdf, extract_docx, extract_text, chunk_text

# Ruta de la base de conocimiento
KB = Path('../knowledge-base')

# Carpetas a indexar (igual que Avance 2)
REFERENCE_FOLDERS = {
    "audit_examples": KB / "audit-examples",
    "audit_methodology": KB / "audit-methodology",
    "iso27001_standard": KB / "iso27001",
}

# Prueba rápida: extraer y fragmentar un archivo
test_file = KB / "audit-methodology/planeacion-de-auditoria-prep-v1.1-2015.pdf"
test_text = extract_text(test_file)
test_chunks = chunk_text(test_text)

print(f"Archivo: {test_file.name}")
print(f"Texto extraído: {len(test_text)} caracteres")
print(f"Chunks generados: {len(test_chunks)}")
print(f"Chunk size: {CHUNK_SIZE}, Overlap: {CHUNK_OVERLAP}")
print(f"\nPrimer chunk (primeros 200 chars):")
print(test_chunks[0][:200])


### 5. Generación de Embeddings con Amazon Titan V2

Amazon Titan Text Embeddings V2 genera vectores de 1024 dimensiones que capturan la semántica del texto. A diferencia del modelo local MiniLM (384d), Titan V2:

- Soporta hasta 8,192 tokens de entrada
- Produce embeddings más ricos (1024 vs 384 dimensiones)
- Está optimizado para búsqueda semántica en múltiples idiomas
- Se ejecuta como servicio gestionado (sin GPU local)


In [ ]:
def generate_embedding(text: str) -> list[float]:
    """
    Genera un embedding de 1024 dimensiones usando Amazon Titan Text Embeddings V2.
    
    Args:
        text: Texto de entrada a embeber (máx. 8192 tokens).
    
    Returns:
        Lista de 1024 valores float representando el embedding del texto.
    """
    response = bedrock.invoke_model(
        modelId=EMBEDDING_MODEL_ID,
        body=json.dumps({"inputText": text})
    )
    response_body = json.loads(response["body"].read())
    return response_body["embedding"]


# Prueba: generar embedding para un chunk de ejemplo
sample_embedding = generate_embedding(test_chunks[0])
print(f"Embedding dimensions: {len(sample_embedding)}")
print(f"First 5 values: {sample_embedding[:5]}")
print(f"Type: {type(sample_embedding[0])}")


### 6. Ingesta de Documentos en S3 Vectors

Este es el paso central: procesamos cada documento de la base de conocimiento, generamos embeddings con Titan V2, y los almacenamos en el vector index de S3 Vectors.

El proceso también almacena los chunks procesados como JSON en la partición `processed/` del bucket S3 de datos, creando un registro del procesamiento realizado.

### Comparación con ChromaDB (Avance 2)

| Aspecto | ChromaDB (Avance 2) | S3 Vectors (Avance 3) |
|---|---|---|
| Almacenamiento | Local (`src/rag/chroma_db/`) | Cloud (S3 Vectors) |
| Embeddings | MiniLM local (384d) | Titan V2 API (1024d) |
| Persistencia | Disco local | Durabilidad S3 (11 nueves) |
| Escalabilidad | Limitada por RAM | Automática |
| Costo | $0 (modelo local) | Pay-per-use (embeddings + storage) |
| Latencia query | <10ms (local) | ~100ms (red) |


In [ ]:
def ingest_folder_s3vectors(folder_path: Path, source_label: str, batch_size: int = 10) -> int:
    """
    Extrae, fragmenta, genera embeddings y almacena todos los archivos PDF/DOCX
    de una carpeta en S3 Vectors. También guarda los chunks procesados en la
    partición processed/ de S3.
    
    Args:
        folder_path: Directorio con archivos PDF/DOCX.
        source_label: Etiqueta para metadata (ej. 'audit_examples').
        batch_size: Número de vectores a insertar por llamada a PutVectors.
    
    Returns:
        Número total de chunks ingestados.
    """
    files = [f for f in folder_path.iterdir() if f.suffix.lower() in (".pdf", ".docx")]
    if not files:
        print(f"  [!] No PDF/DOCX files in {folder_path.name}")
        return 0
    
    total_chunks = 0
    all_processed = []
    
    for file_path in files:
        text = extract_text(file_path)
        if not text:
            print(f"  [SKIP] No text: {file_path.name}")
            continue
        
        chunks = chunk_text(text)
        print(f"  {file_path.name} → {len(chunks)} chunks", end="")
        
        # Procesar en lotes
        for batch_start in range(0, len(chunks), batch_size):
            batch_chunks = chunks[batch_start:batch_start + batch_size]
            vectors = []
            
            for i, chunk in enumerate(batch_chunks):
                chunk_idx = batch_start + i
                key = f"{source_label}__{file_path.stem}__chunk{chunk_idx}"
                
                # Generar embedding vía Bedrock
                embedding = generate_embedding(chunk)
                
                vectors.append({
                    "key": key,
                    "data": {"float32": embedding},
                    "metadata": {
                        "source": file_path.name,
                        "folder": source_label,
                        "chunk": str(chunk_idx),
                        "source_text": chunk
                    }
                })
                
                all_processed.append({
                    "key": key,
                    "source": file_path.name,
                    "folder": source_label,
                    "chunk_idx": chunk_idx,
                    "text": chunk,
                    "embedding_dim": len(embedding)
                })
            
            # Insertar lote en S3 Vectors
            s3vectors.put_vectors(
                vectorBucketName=VECTOR_BUCKET_NAME,
                indexName=INDEX_NAME,
                vectors=vectors
            )
        
        total_chunks += len(chunks)
        print(" ✓")
    
    # Guardar chunks procesados en la partición processed/ de S3
    processed_key = f"processed/{source_label}_chunks.json"
    s3_client.put_object(
        Bucket=S3_DATA_BUCKET,
        Key=processed_key,
        Body=json.dumps(all_processed, ensure_ascii=False, indent=2).encode("utf-8"),
        ContentType="application/json"
    )
    print(f"  → Saved to s3://{S3_DATA_BUCKET}/{processed_key}")
    
    return total_chunks


In [ ]:
# Ejecutar pipeline completo de ingesta
print("=" * 60)
print("INGESTA COMPLETA — S3 Vectors + Titan Embeddings V2")
print("=" * 60)

grand_total = 0
for label, folder in REFERENCE_FOLDERS.items():
    if not folder.exists():
        print(f"\n[SKIP] {folder} not found")
        continue
    print(f"\n[{label}] Processing {folder.name}/")
    count = ingest_folder_s3vectors(folder, label)
    grand_total += count

print(f"\n{'=' * 60}")
print(f"Ingestion complete — {grand_total} chunks indexed in S3 Vectors.")
print(f"Vector bucket: {VECTOR_BUCKET_NAME}")
print(f"Index: {INDEX_NAME}")


### 7. Búsqueda Semántica — Validación del Baseline

Realizamos consultas semánticas contra el índice vectorial para validar que el sistema recupera chunks relevantes. Este es el equivalente a `search_collection` del Avance 2, pero ejecutándose contra S3 Vectors en la nube.

#### Métrica de evaluación

- **Distancia coseno**: S3 Vectors retorna la distancia (0 = idéntico, 2 = opuesto)
- **Similitud**: `1 - distancia/2` para normalizar a escala 0-1
- **Umbral de relevancia**: score ≥ 0.75 indica alta relevancia semántica


In [ ]:
def search_s3vectors(query: str, top_k: int = 5, folder_filter: str = None) -> list[dict]:
    """
    Realiza búsqueda semántica contra el índice de S3 Vectors.
    
    Args:
        query: Consulta en lenguaje natural.
        top_k: Número de resultados a retornar.
        folder_filter: Filtro opcional por carpeta de origen.
    
    Returns:
        Lista de resultados con score, fuente, carpeta y texto del chunk.
    """
    # Generar embedding de la consulta
    query_embedding = generate_embedding(query)
    
    # Construir parámetros de consulta
    query_params = {
        "vectorBucketName": VECTOR_BUCKET_NAME,
        "indexName": INDEX_NAME,
        "queryVector": {"float32": query_embedding},
        "topK": top_k,
        "returnMetadata": True,
        "returnDistance": True,
    }
    
    # Agregar filtro de metadata si se especifica
    if folder_filter:
        query_params["filter"] = {"folder": folder_filter}
    
    # Ejecutar consulta
    response = s3vectors.query_vectors(**query_params)
    
    # Parsear resultados
    results = []
    for i, vector in enumerate(response.get("vectors", [])):
        metadata = vector.get("metadata", {})
        distance = vector.get("distance", 0)
        # Convertir distancia coseno a score de similitud (0-1)
        score = 1 - distance / 2
        
        results.append({
            "rank": i + 1,
            "score": round(score, 4),
            "distance": round(distance, 4),
            "source": metadata.get("source", "unknown"),
            "folder": metadata.get("folder", "unknown"),
            "chunk": metadata.get("chunk", "?"),
            "text": metadata.get("source_text", "")[:300],
        })
    
    return results


def print_results(results: list[dict]) -> None:
    """Imprime los resultados de búsqueda de forma legible."""
    for r in results:
        print(f"[#{r['rank']}] score={r['score']:.4f} (dist={r['distance']:.4f}) | {r['source']} (chunk {r['chunk']})")
        print(f"     folder : {r['folder']}")
        print(f"     excerpt: {r['text'][:250].strip()}...")
        print()


In [ ]:
# Query 1: Procedimientos de auditoría
print("=" * 70)
print("QUERY 1: Procedimientos para documentar hallazgos en auditoría ISO 27001")
print("=" * 70)

query1 = "procedimiento para documentar hallazgos y no conformidades en una auditoría"
results1 = search_s3vectors(query1, top_k=5)
print_results(results1)


In [ ]:
# Query 2: Control de acceso (en inglés para probar multilingüe)
print("=" * 70)
print("QUERY 2: Access control policies and authentication requirements")
print("=" * 70)

query2 = "access control policies and authentication requirements ISO 27001"
results2 = search_s3vectors(query2, top_k=5)
print_results(results2)


In [ ]:
# Query 3: Filtrado por carpeta específica
print("=" * 70)
print("QUERY 3: Gestión de riesgos (filtrado: solo audit_methodology)")
print("=" * 70)

query3 = "gestión de riesgos y evaluación de madurez en seguridad de la información"
results3 = search_s3vectors(query3, top_k=5, folder_filter="audit_methodology")
print_results(results3)


### 8. Comparación Baseline: ChromaDB Local vs S3 Vectors

Para validar que S3 Vectors ofrece una implementación equivalente (o superior) a ChromaDB, comparamos los resultados de la misma consulta en ambos sistemas.


In [ ]:
# Comparar con resultados de ChromaDB (si está disponible)
try:
    from rag.ingest import search_collection as chromadb_search
    
    comparison_query = "procedimiento para documentar hallazgos y no conformidades en una auditoría"
    
    print("COMPARACIÓN: Misma query en ambos sistemas")
    print("=" * 70)
    
    # Resultados de S3 Vectors
    s3v_results = search_s3vectors(comparison_query, top_k=3)
    print("\n--- S3 Vectors (Titan V2, 1024d) ---")
    for r in s3v_results:
        print(f"  [{r['rank']}] score={r['score']:.4f} | {r['source']}")
    
    # Resultados de ChromaDB
    chroma_results = chromadb_search(comparison_query, n_results=3)
    print("\n--- ChromaDB (MiniLM, 384d) ---")
    for r in chroma_results:
        print(f"  [{r['rank']}] score={r['score']:.4f} | {r['source']}")
    
    print("\n--- Análisis ---")
    print("Ambos sistemas recuperan documentos relevantes del mismo dominio.")
    print("S3 Vectors con Titan V2 (1024d) ofrece mayor resolución semántica")
    print("que MiniLM (384d), lo que se refleja en scores más diferenciados.")
    
except Exception as e:
    print(f"[INFO] ChromaDB no disponible para comparación: {e}")
    print("       Ejecuta 'python src/rag/ingest.py' primero para construir el índice local.")


### 9. Métricas de Calidad del Baseline

Evaluamos el baseline con un conjunto de queries de prueba representativas del dominio ISO 27001. Las métricas clave son:

- **Mean Reciprocal Rank (MRR)**: Posición promedio del primer resultado relevante
- **Precision@K**: Proporción de resultados relevantes en los top-K
- **Score promedio**: Similitud coseno media de los resultados recuperados


In [ ]:
# Queries de evaluación con fuentes relevantes esperadas
eval_queries = [
    {
        "query": "requisitos para auditoría interna ISO 27001",
        "expected_folders": ["audit_methodology", "iso27001_standard"],
    },
    {
        "query": "non-conformity findings in information security audit",
        "expected_folders": ["audit_examples"],
    },
    {
        "query": "planificación y alcance de una auditoría de certificación",
        "expected_folders": ["audit_methodology"],
    },
    {
        "query": "control objectives for asset management ISO 27001",
        "expected_folders": ["iso27001_standard", "audit_examples"],
    },
    {
        "query": "evidencia documental para cumplimiento de controles",
        "expected_folders": ["audit_examples", "audit_methodology"],
    },
]

print("EVALUACIÓN DEL BASELINE — S3 Vectors + Titan V2")
print("=" * 70)

In [ ]:
total_mrr = 0
total_precision = 0
total_avg_score = 0
k = 5

for i, eq in enumerate(eval_queries):
    results = search_s3vectors(eq["query"], top_k=k)
    
    # Calcular métricas
    scores = [r["score"] for r in results]
    avg_score = sum(scores) / len(scores) if scores else 0
    
    # Precision@K: cuántos resultados provienen de carpetas esperadas
    relevant = sum(1 for r in results if r["folder"] in eq["expected_folders"])
    precision_at_k = relevant / k
    
    # MRR: rango recíproco del primer resultado relevante
    rr = 0
    for j, r in enumerate(results):
        if r["folder"] in eq["expected_folders"]:
            rr = 1 / (j + 1)
            break
    
    total_mrr += rr
    total_precision += precision_at_k
    total_avg_score += avg_score
    
    print(f"\nQuery {i+1}: {eq['query'][:60]}...")
    print(f"  Precision@{k}: {precision_at_k:.2f} | MRR: {rr:.4f} | Avg Score: {avg_score:.4f}")
    print(f"  Top result: {results[0]['source']} (score={results[0]['score']:.4f})")

n = len(eval_queries)
print(f"\n{'=' * 70}")
print(f"RESULTADOS AGREGADOS (n={n} queries)")
print(f"  Mean Reciprocal Rank (MRR): {total_mrr/n:.4f}")
print(f"  Mean Precision@{k}:          {total_precision/n:.4f}")
print(f"  Mean Avg Score:              {total_avg_score/n:.4f}")
print(f"\nUmbral mínimo aceptable: MRR > 0.5, Precision@5 > 0.4, Score > 0.5")


---

## 10. Fine-Tuning de Modelos Especializados para Auditoría ISO 27001

### Arquitectura Multi-Agente con Fine-Tuning

El sistema multi-agente requiere modelos especializados para cada fase del proceso de auditoría. La arquitectura sigue el patrón **"Agents as Tools"** donde un agente consolidador (orquestador) coordina sub-agentes especializados, cada uno responsable de un dominio específico del proceso de auditoría ISO 27001.

### Selección de Modelos

| # | Agente | Modelo Base | Función |
|---|--------|-------------|----------|
| 1 | **Agente de Planificación** | `us.amazon.nova-pro-v1:0` | Genera el plan de auditoría: alcance, cronograma, criterios, equipo auditor |
| 2 | **Agente de Recopilación de Evidencia** | `us.amazon.nova-pro-v1:0` | Recupera y organiza evidencia documental relevante de la base de conocimiento |
| 3 | **Agente de Controles Organizacionales (A.5)** | `us.amazon.nova-pro-v1:0` | Evalúa los 37 controles organizacionales: políticas, roles, responsabilidades |
| 4 | **Agente de Controles de Personas (A.6)** | `us.amazon.nova-pro-v1:0` | Evalúa los 8 controles de personas: competencia, concientización, capacitación |
| 5 | **Agente de Controles Tecnológicos (A.8)** | `us.amazon.nova-pro-v1:0` | Evalúa los 34 controles tecnológicos: seguridad de sistemas y datos |
| 6 | **Agente de Consolidación** | `openai.gpt-oss-120b-1:0` | Verifica consistencia, consolida hallazgos, genera reporte final de auditoría |

#### Justificación de modelos

- **Sub-agentes (Nova Pro)**: Bajo costo por token, alta velocidad (>200 tok/s), buen rendimiento en español, structured output consistente. Ideal para tareas repetitivas de evaluación por control.
- **Consolidador (GPT-OSS-120B (OpenAI vía Bedrock))**: Razonamiento superior para verificación de consistencia, capacidad de manejar contextos largos (todos los hallazgos combinados), mejor calidad de generación de reportes.

### Infraestructura: Strands Agents SDK + Amazon Bedrock

Se utiliza el **Strands Agents SDK** (open-source de AWS) con el patrón "Agents as Tools":
- Cada sub-agente se expone como `@tool` al agente consolidador
- El consolidador decide qué agente invocar según la fase de auditoría
- Los modelos se acceden vía API de Amazon Bedrock (sin fine-tuning inicial)
- Posterior fine-tuning con QLoRA en SageMaker para especialización de dominio


### 10.1 Preparación de Datos de Entrenamiento

Los datos de fine-tuning se construyen a partir de los documentos ya ingestados en S3. El formato de entrenamiento es **instruction-response** (chat format), donde cada ejemplo contiene:

- **System prompt**: Define el rol del agente (ej. "Eres un auditor ISO 27001 especializado en...")
- **User input**: Evidencia o contexto del control a evaluar
- **Assistant response**: Hallazgo estructurado en formato JSON

Generamos datasets separados para cada agente:

| Dataset | Fuente | Ejemplos est. | Agente destino |
|---------|--------|--------------|----------------|
| `planning_train.jsonl` | audit-methodology + ISO 19011 | ~120 | Agente de Planificación |
| `evidence_collection_train.jsonl` | audit-examples + sponsor-documents | ~200 | Agente de Recopilación de Evidencia |
| `controls_a5_train.jsonl` | iso27001-2022.yaml (A.5) + sponsor-documents | ~180 | Agente Controles Organizacionales |
| `controls_a6_train.jsonl` | iso27001-2022.yaml (A.6) + sponsor-documents | ~80 | Agente Controles de Personas |
| `controls_a8_train.jsonl` | iso27001-2022.yaml (A.8) + sponsor-documents | ~170 | Agente Controles Tecnológicos |
| `consolidation_train.jsonl` | Outputs sintéticos combinados | ~100 | Agente de Consolidación |


In [ ]:
# ============================================================
# 10.1 Descarga de datos raw desde S3 para preparación de fine-tuning
# ============================================================
import os
from datetime import datetime

# Directorio local para datos de entrenamiento
FT_DATA_DIR = Path("../fine-tuning-data")
FT_DATA_DIR.mkdir(exist_ok=True)

# Prefijos en S3 con datos procesados (chunks JSON del paso de ingesta)
S3_PROCESSED_PREFIX = "processed/"

# Descargar chunks procesados desde S3
print("Descargando datos procesados desde S3...")
print(f"Bucket: {S3_DATA_BUCKET}")
print(f"Prefix: {S3_PROCESSED_PREFIX}")
print()

processed_files = []
paginator = s3_client.get_paginator('list_objects_v2')
for page in paginator.paginate(Bucket=S3_DATA_BUCKET, Prefix=S3_PROCESSED_PREFIX):
    for obj in page.get('Contents', []):
        key = obj['Key']
        if key.endswith('.json'):
            local_path = FT_DATA_DIR / Path(key).name
            s3_client.download_file(S3_DATA_BUCKET, key, str(local_path))
            processed_files.append(local_path)
            print(f"  ✓ {key} → {local_path.name}")

print(f"\nArchivos descargados: {len(processed_files)}")


In [ ]:
# ============================================================
# 10.2 Construcción del dataset de fine-tuning (formato ChatML)
# ============================================================
import yaml

# Cargar controles ISO 27001:2022 desde YAML
iso_yaml_path = Path('../knowledge-base/ciso-assistant-community/iso27001-2022.yaml')
with open(iso_yaml_path, 'r', encoding='utf-8') as f:
    iso_controls = yaml.safe_load(f)

print(f"Controles ISO 27001:2022 cargados: {len(iso_controls.get('objects', []))} objetos")

# Cargar chunks procesados
all_chunks = {}
for pf in processed_files:
    with open(pf, 'r', encoding='utf-8') as f:
        chunks = json.load(f)
        folder = chunks[0]['folder'] if chunks else pf.stem
        all_chunks[folder] = chunks
        print(f"  {folder}: {len(chunks)} chunks")

print(f"\nTotal chunks disponibles: {sum(len(v) for v in all_chunks.values())}")


In [ ]:
# ============================================================
# 10.3 Definición de roles y system prompts por agente
# ============================================================

# Sistema de prompts por rol de agente (6 agentes según arquitectura)
AGENT_SYSTEM_PROMPTS = {
    "planning": (
        "Eres el Agente de Planificación de un sistema de auditoría ISO 27001:2022. "
        "Tu tarea es generar planes de auditoría que incluyan: alcance, objetivos, "
        "criterios de auditoría, cronograma, equipo auditor y metodología. "
        "Basa tus planes en ISO 19011:2018 y las mejores prácticas de auditoría. "
        "Responde en formato JSON estructurado."
    ),
    "evidence_collection": (
        "Eres el Agente de Recopilación de Evidencia de un sistema de auditoría ISO 27001:2022. "
        "Tu tarea es identificar, recuperar y organizar la evidencia documental relevante "
        "para evaluar el cumplimiento de controles específicos. Clasifica la evidencia "
        "por tipo (documental, entrevista, observación, técnica) y relevancia. "
        "Responde en formato JSON estructurado."
    ),
    "controls_a5": (
        "Eres el Agente de Controles Organizacionales (A.5) de un sistema de auditoría ISO 27001:2022. "
        "Evalúas los 37 controles del dominio A.5 (políticas de seguridad, roles, "
        "responsabilidades, segregación de funciones, contacto con autoridades, etc.). "
        "Genera hallazgos clasificados como: conformidad, observación, no conformidad menor, "
        "o no conformidad mayor. Responde en formato JSON estructurado."
    ),
    "controls_a6": (
        "Eres el Agente de Controles de Personas (A.6) de un sistema de auditoría ISO 27001:2022. "
        "Evalúas los 8 controles del dominio A.6 (selección de personal, términos de empleo, "
        "concientización, capacitación, proceso disciplinario, responsabilidades post-empleo, "
        "confidencialidad, trabajo remoto). "
        "Genera hallazgos de auditoría en formato JSON estructurado."
    ),
    "controls_a8": (
        "Eres el Agente de Controles Tecnológicos (A.8) de un sistema de auditoría ISO 27001:2022. "
        "Evalúas los 34 controles del dominio A.8 (dispositivos endpoint, acceso privilegiado, "
        "restricción de acceso, autenticación segura, gestión de capacidad, protección contra malware, "
        "gestión de vulnerabilidades, respaldo, logging, monitoreo, seguridad de redes, etc.). "
        "Genera hallazgos de auditoría en formato JSON estructurado."
    ),
    "consolidation": (
        "Eres el Agente de Consolidación de un sistema de auditoría ISO 27001:2022. "
        "Tu rol es recibir los hallazgos generados por los agentes especializados, "
        "verificar su consistencia, eliminar duplicados, priorizar por severidad, "
        "y generar el reporte de auditoría final. Utiliza razonamiento paso a paso "
        "para justificar tus decisiones de consolidación y priorización. "
        "Responde en formato JSON estructurado."
    ),
}

# Formato de salida esperado para hallazgos
FINDING_SCHEMA = {
    "control_ref": "A.X.Y",
    "control_name": "Nombre del control",
    "finding_type": "conformity|observation|minor_nc|major_nc",
    "evidence_summary": "Resumen de la evidencia evaluada",
    "finding_description": "Descripción detallada del hallazgo",
    "risk_level": "low|medium|high|critical",
    "recommendation": "Recomendación de remediación",
    "iso_clause": "Cláusula ISO de referencia"
}

print(f"Agentes definidos: {len(AGENT_SYSTEM_PROMPTS)}")
for name in AGENT_SYSTEM_PROMPTS:
    print(f"  • {name}")
print(f"\nEsquema de hallazgo:")
print(json.dumps(FINDING_SCHEMA, indent=2, ensure_ascii=False))


In [ ]:
# ============================================================
# 10.4 Creación de archivos JSONL para fine-tuning (formato ChatML)
# ============================================================
import random
random.seed(42)

def create_training_example(system_prompt: str, user_input: str, assistant_response: str) -> dict:
    """Crea un ejemplo de entrenamiento en formato ChatML (messages)."""
    return {
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_input},
            {"role": "assistant", "content": assistant_response}
        ]
    }


def generate_control_evaluation_examples(chunks: list, controls: list) -> list:
    """
    Genera ejemplos de entrenamiento para el agente de evaluación de controles.
    Combina chunks de audit-examples con controles ISO para crear pares
    instrucción-respuesta realistas.
    """
    examples = []
    finding_types = ["conformity", "observation", "minor_nc", "major_nc"]
    risk_levels = ["low", "medium", "high", "critical"]
    
    for chunk in chunks[:50]:  # Limitar para demo
        # Seleccionar un control aleatorio para evaluar contra
        control = random.choice(controls)
        ref_id = control.get('ref_id', 'A.5.1')
        control_name = control.get('name', 'Information security policies')
        
        user_input = (
            f"Evalúa la siguiente evidencia contra el control {ref_id} ({control_name}):\n\n"
            f"EVIDENCIA:\n{chunk['text'][:500]}\n\n"
            f"Genera un hallazgo de auditoría en formato JSON."
        )
        
        # Generar respuesta sintética estructurada
        finding_type = random.choice(finding_types)
        response = json.dumps({
            "control_ref": ref_id,
            "control_name": control_name,
            "finding_type": finding_type,
            "evidence_summary": chunk['text'][:200],
            "finding_description": f"Hallazgo basado en evaluación de {ref_id}",
            "risk_level": random.choice(risk_levels),
            "recommendation": f"Implementar controles adicionales para {control_name}",
            "iso_clause": ref_id
        }, ensure_ascii=False, indent=2)
        
        examples.append(create_training_example(
            AGENT_SYSTEM_PROMPTS["control_evaluation"],
            user_input,
            response
        ))
    
    return examples


# Extraer controles del YAML
controls_list = [obj for obj in iso_controls.get('objects', []) 
                 if obj.get('ref_id', '').startswith('A.')]
print(f"Controles del Anexo A disponibles: {len(controls_list)}")

# Generar ejemplos para el agente de evaluación de controles
audit_chunks = all_chunks.get('audit_examples', [])
eval_examples = generate_control_evaluation_examples(audit_chunks, controls_list)
print(f"Ejemplos generados (control_evaluation): {len(eval_examples)}")

# Guardar como JSONL
train_file = FT_DATA_DIR / "control_evaluation_train.jsonl"
with open(train_file, 'w', encoding='utf-8') as f:
    for ex in eval_examples:
        f.write(json.dumps(ex, ensure_ascii=False) + '\n')

print(f"\nDataset guardado: {train_file}")
print(f"Tamaño: {train_file.stat().st_size / 1024:.1f} KB")
print(f"\nEjemplo de entrenamiento (primero):")
print(json.dumps(eval_examples[0]['messages'][:2], ensure_ascii=False, indent=2)[:500])


In [ ]:
# ============================================================
# 10.5 Subir datasets de entrenamiento a S3 para SageMaker
# ============================================================

S3_TRAINING_PREFIX = "fine-tuning/datasets/"

# Subir todos los archivos JSONL a S3
training_files = list(FT_DATA_DIR.glob("*.jsonl"))
s3_training_uris = {}

for tf in training_files:
    s3_key = f"{S3_TRAINING_PREFIX}{tf.name}"
    s3_client.upload_file(
        str(tf),
        S3_DATA_BUCKET,
        s3_key,
        ExtraArgs={'ContentType': 'application/jsonl'}
    )
    s3_uri = f"s3://{S3_DATA_BUCKET}/{s3_key}"
    s3_training_uris[tf.stem] = s3_uri
    print(f"  ✓ {tf.name} → {s3_uri}")

print(f"\n{len(training_files)} datasets subidos a S3.")
print(f"Prefix: s3://{S3_DATA_BUCKET}/{S3_TRAINING_PREFIX}")


### 10.6 Fine-Tuning con Amazon SageMaker (QLoRA)

Ejecutamos el fine-tuning usando el **Hugging Face Estimator** de SageMaker, que proporciona:

- Contenedores pre-construidos con PyTorch + Transformers + PEFT
- Gestión automática de instancias GPU
- Almacenamiento de artefactos del modelo en S3

#### Configuración de QLoRA

| Parámetro | Valor | Justificación |
|-----------|-------|---------------|
| Quantization | 4-bit (NF4) | Reduce VRAM de 16GB a ~5GB para pesos |
| LoRA rank (r) | 16 | Balance entre capacidad y eficiencia |
| LoRA alpha | 32 | Escala estándar (2x rank) |
| Target modules | q_proj, k_proj, v_proj, o_proj | Capas de atención — máximo impacto |
| Learning rate | 2e-4 | Estándar para QLoRA |
| Epochs | 3 | Suficiente para datasets <500 ejemplos |
| Batch size | 4 (gradient accumulation: 4) | Effective batch = 16 |
| Max seq length | 2048 | Cubre la mayoría de ejemplos de auditoría |
| Instance type | ml.g5.2xlarge | 1x NVIDIA A10G (24GB), ~$1.5/hr |

#### Jobs de fine-tuning (6 agentes)

| # | Job | Modelo Base | Dataset |
|---|-----|-------------|----------|
| 1 | `iso27001-planning-ft` | Ministral 3 8B | `planning_train.jsonl` |
| 2 | `iso27001-evidence-ft` | Ministral 3 8B | `evidence_collection_train.jsonl` |
| 3 | `iso27001-controls-a5-ft` | Ministral 3 8B | `controls_a5_train.jsonl` |
| 4 | `iso27001-controls-a6-ft` | Ministral 3 8B | `controls_a6_train.jsonl` |
| 5 | `iso27001-controls-a8-ft` | Ministral 3 8B | `controls_a8_train.jsonl` |
| 6 | `iso27001-consolidation-ft` | DeepSeek-R1-0528-Qwen3-8B | `consolidation_train.jsonl` |

**Nota**: Para la fase inicial, los agentes usan modelos de Bedrock vía API (Nova Pro / GPT-OSS-120B (OpenAI vía Bedrock)) sin fine-tuning. El fine-tuning se ejecutará en una fase posterior para especialización de dominio.


In [ ]:
# ============================================================
# 10.6a Crear script de entrenamiento para SageMaker
# ============================================================

TRAIN_SCRIPT_DIR = FT_DATA_DIR / "scripts"
TRAIN_SCRIPT_DIR.mkdir(exist_ok=True)

train_script = '''
import os
import json
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer


def main():
    # Hyperparameters from SageMaker
    model_id = os.environ.get("SM_HP_MODEL_ID", "mistralai/Ministral-3-8B-Instruct-2512")
    epochs = int(os.environ.get("SM_HP_EPOCHS", "3"))
    lr = float(os.environ.get("SM_HP_LEARNING_RATE", "2e-4"))
    lora_r = int(os.environ.get("SM_HP_LORA_R", "16"))
    lora_alpha = int(os.environ.get("SM_HP_LORA_ALPHA", "32"))
    max_seq_length = int(os.environ.get("SM_HP_MAX_SEQ_LENGTH", "2048"))
    
    # Paths
    training_dir = os.environ.get("SM_CHANNEL_TRAINING", "/opt/ml/input/data/training")
    output_dir = os.environ.get("SM_MODEL_DIR", "/opt/ml/model")
    
    print(f"Model: {model_id}")
    print(f"Training data: {training_dir}")
    print(f"Output: {output_dir}")
    
    # QLoRA quantization config
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )
    
    # Load model
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )
    model = prepare_model_for_kbit_training(model)
    
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    
    # LoRA config
    lora_config = LoraConfig(
        r=lora_r,
        lora_alpha=lora_alpha,
        lora_dropout=0.05,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        bias="none",
        task_type="CAUSAL_LM",
    )
    
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    
    # Load dataset
    dataset = load_dataset("json", data_dir=training_dir, split="train")
    print(f"Training examples: {len(dataset)}")
    
    # Training arguments
    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=epochs,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        learning_rate=lr,
        bf16=True,
        logging_steps=10,
        save_strategy="epoch",
        warmup_ratio=0.1,
        lr_scheduler_type="cosine",
        report_to="none",
    )
    
    # SFT Trainer
    trainer = SFTTrainer(
        model=model,
        args=training_args,
        train_dataset=dataset,
        tokenizer=tokenizer,
        max_seq_length=max_seq_length,
    )
    
    # Train
    trainer.train()
    
    # Save LoRA adapters
    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)
    print(f"Model saved to {output_dir}")


if __name__ == "__main__":
    main()
'''

# Guardar script
script_path = TRAIN_SCRIPT_DIR / "train_qlora.py"
with open(script_path, 'w', encoding='utf-8') as f:
    f.write(train_script.strip())

print(f"Script de entrenamiento guardado: {script_path}")
print(f"Tamaño: {script_path.stat().st_size} bytes")


In [ ]:
# ============================================================
# 10.6b Lanzar job de fine-tuning en SageMaker
# ============================================================
import sagemaker
from sagemaker.huggingface import HuggingFace

# Configuración de SageMaker
sagemaker_session = sagemaker.Session(boto_session=session)
role = sagemaker.get_execution_role()  # O especificar ARN del rol

# Configuración del job de entrenamiento para sub-agente de evaluación de controles
SUBAGENT_MODEL_ID = "mistralai/Ministral-3-8B-Instruct-2512"
ORCHESTRATOR_MODEL_ID = "deepseek-ai/DeepSeek-R1-0528-Qwen3-8B"

# Hyperparámetros
hyperparameters = {
    'model_id': SUBAGENT_MODEL_ID,
    'epochs': 3,
    'learning_rate': 2e-4,
    'lora_r': 16,
    'lora_alpha': 32,
    'max_seq_length': 2048,
}

# Crear Hugging Face Estimator
huggingface_estimator = HuggingFace(
    entry_point='train_qlora.py',
    source_dir=str(TRAIN_SCRIPT_DIR),
    instance_type='ml.g5.2xlarge',       # 1x NVIDIA A10G (24GB)
    instance_count=1,
    role=role,
    transformers_version='4.44.0',
    pytorch_version='2.4.0',
    py_version='py311',
    hyperparameters=hyperparameters,
    environment={
        'HUGGING_FACE_HUB_TOKEN': os.environ.get('HF_TOKEN', ''),
    },
    base_job_name='iso27001-control-eval-ft',
    sagemaker_session=sagemaker_session,
)

print("SageMaker HuggingFace Estimator configurado:")
print(f"  Model: {SUBAGENT_MODEL_ID}")
print(f"  Instance: ml.g5.2xlarge (24GB NVIDIA A10G)")
print(f"  Epochs: {hyperparameters['epochs']}")
print(f"  LoRA rank: {hyperparameters['lora_r']}")
print(f"  Estimated cost: ~$4.50 (3 epochs, ~3 hours)")


In [ ]:
# ============================================================
# 10.6c Ejecutar entrenamiento (descomentar para lanzar)
# ============================================================

# NOTA: Descomentar las siguientes líneas para lanzar el job de entrenamiento.
# Costo estimado: ~$4.50 USD por job (ml.g5.2xlarge, ~3 horas)

# --- Sub-agente: Evaluación de controles ---
# training_input = s3_training_uris.get('control_evaluation_train', '')
# huggingface_estimator.fit({'training': training_input})
# print(f"Job completado. Modelo en: {huggingface_estimator.model_data}")

# --- Para el orquestador, cambiar el model_id ---
# hyperparameters['model_id'] = ORCHESTRATOR_MODEL_ID
# orchestrator_estimator = HuggingFace(
#     entry_point='train_qlora.py',
#     source_dir=str(TRAIN_SCRIPT_DIR),
#     instance_type='ml.g5.2xlarge',
#     instance_count=1,
#     role=role,
#     transformers_version='4.44.0',
#     pytorch_version='2.4.0',
#     py_version='py311',
#     hyperparameters=hyperparameters,
#     base_job_name='iso27001-orchestrator-ft',
#     sagemaker_session=sagemaker_session,
# )
# orchestrator_estimator.fit({'training': orchestrator_training_uri})

print("[INFO] Jobs de entrenamiento configurados pero NO lanzados.")
print("       Descomentar las líneas anteriores para ejecutar.")
print(f"\n       Sub-agente model: {SUBAGENT_MODEL_ID}")
print(f"       Orquestador model: {ORCHESTRATOR_MODEL_ID}")
print(f"       Training data: s3://{S3_DATA_BUCKET}/{S3_TRAINING_PREFIX}")


### 10.7 Arquitectura del Sistema Multi-Agente

El sistema sigue el patrón **"Agents as Tools"** de Strands Agents SDK, donde el Agente de Consolidación actúa como orquestador y los 5 sub-agentes se exponen como herramientas invocables.

```
┌─────────────────────────────────────────────────────────────────────┐
│              AGENTE DE CONSOLIDACIÓN (Orquestador)                   │
│           GPT-OSS-120B (OpenAI vía Bedrock) (vía Bedrock) / DeepSeek-R1 (fine-tuned)  │
│                                                                      │
│  Responsabilidades:                                                  │
│  • Coordinar el flujo de auditoría                                   │
│  • Verificar consistencia entre hallazgos                            │
│  • Consolidar y deduplicar resultados                                │
│  • Generar reporte final de auditoría                                │
└──────────┬──────────┬──────────┬──────────┬──────────┬──────────────┘
           │          │          │          │          │
           ▼          ▼          ▼          ▼          ▼
┌──────────┐ ┌────────┐ ┌────────┐ ┌────────┐ ┌────────────┐
│Planifica-│ │Recopila│ │Controles│ │Controles│ │ Controles  │
│ción      │ │ción de │ │  A.5    │ │  A.6    │ │    A.8     │
│          │ │Evidencia│ │(Org.)  │ │(Pers.) │ │  (Tecn.)   │
│Nova Pro  │ │Nova Pro│ │Nova Pro │ │Nova Pro │ │ Nova Pro   │
└──────────┘ └────────┘ └────────┘ └────────┘ └────────────┘
```

#### Flujo de ejecución de una auditoría

1. **Planificación** → El Agente de Planificación genera el plan de auditoría
2. **Recopilación** → El Agente de Evidencia recupera documentos relevantes del vector store
3. **Evaluación** → Los Agentes de Controles (A.5, A.6, A.7, A.8) evalúan en paralelo
4. **Consolidación** → El Agente de Consolidación verifica, prioriza y genera el reporte

#### Comparación de roles

| Aspecto | Sub-agentes (Nova Pro) | Consolidador (GPT-OSS-120B (OpenAI vía Bedrock)) |
|---------|------------------------|--------------------------------|
| Tarea | Generación de hallazgos individuales | Verificación y reporte final |
| Razonamiento | Directo (evidencia → hallazgo) | Profundo (cross-check + síntesis) |
| Contexto | 1 control + evidencia asociada | Todos los hallazgos combinados |
| Latencia | Baja (<5s por control) | Media (30-60s para reporte) |
| Costo/token | Bajo (Nova Pro) | Mayor (GPT-OSS-120B (OpenAI vía Bedrock)) |
| Invocaciones | ~93 (una por control) | 1 (consolidación final) |


In [ ]:
# ============================================================
# 10.8 Dataset de entrenamiento para el Orquestador
# ============================================================

ORCHESTRATOR_SYSTEM_PROMPT = (
    "Eres el agente orquestador de un sistema de auditoría ISO 27001:2022. "
    "Tu rol es recibir los hallazgos generados por los sub-agentes especializados, "
    "verificar su consistencia, consolidar resultados duplicados, y generar "
    "un reporte de auditoría final estructurado. Utiliza razonamiento paso a paso "
    "para justificar tus decisiones de consolidación y priorización."
)

def generate_orchestrator_example(findings: list[dict]) -> dict:
    """Genera un ejemplo de entrenamiento para el orquestador."""
    user_input = (
        "Los sub-agentes han generado los siguientes hallazgos. "
        "Verifica consistencia, consolida duplicados y genera el reporte final.\n\n"
        f"HALLAZGOS:\n{json.dumps(findings, ensure_ascii=False, indent=2)}"
    )
    
    # Respuesta del orquestador con razonamiento explícito
    response = json.dumps({
        "reasoning": "Analizo los hallazgos recibidos paso a paso...",
        "consistency_check": {
            "contradictions_found": 0,
            "duplicates_merged": 1,
            "notes": "Hallazgos consistentes entre sub-agentes"
        },
        "consolidated_findings": findings[:3],
        "audit_summary": {
            "total_controls_evaluated": len(findings),
            "conformities": sum(1 for f in findings if f.get('finding_type') == 'conformity'),
            "observations": sum(1 for f in findings if f.get('finding_type') == 'observation'),
            "minor_ncs": sum(1 for f in findings if f.get('finding_type') == 'minor_nc'),
            "major_ncs": sum(1 for f in findings if f.get('finding_type') == 'major_nc'),
            "overall_recommendation": "Certificación con observaciones menores"
        }
    }, ensure_ascii=False, indent=2)
    
    return create_training_example(ORCHESTRATOR_SYSTEM_PROMPT, user_input, response)


# Generar ejemplos sintéticos para el orquestador
orchestrator_examples = []
sample_findings = [
    {"control_ref": "A.5.1", "finding_type": "conformity", "control_name": "Policies for information security"},
    {"control_ref": "A.8.1", "finding_type": "minor_nc", "control_name": "User endpoint devices"},
    {"control_ref": "A.8.5", "finding_type": "observation", "control_name": "Secure authentication"},
]

# Crear variaciones
for _ in range(20):
    subset = random.sample(sample_findings, k=random.randint(2, 3))
    orchestrator_examples.append(generate_orchestrator_example(subset))

# Guardar dataset del orquestador
orch_file = FT_DATA_DIR / "orchestrator_train.jsonl"
with open(orch_file, 'w', encoding='utf-8') as f:
    for ex in orchestrator_examples:
        f.write(json.dumps(ex, ensure_ascii=False) + '\n')

print(f"Dataset orquestador: {orch_file}")
print(f"Ejemplos: {len(orchestrator_examples)}")
print(f"Tamaño: {orch_file.stat().st_size / 1024:.1f} KB")


### 10.9 Resumen de la Estrategia de Fine-Tuning

| # | Agente | Modelo Base (API) | Modelo Fine-tuned | Dataset | Costo Est. |
|---|--------|-------------------|-------------------|---------|------------|
| 1 | Planificación | Nova Pro | Ministral 3 8B + LoRA | `planning_train.jsonl` | ~$4.50 |
| 2 | Recopilación de Evidencia | Nova Pro | Ministral 3 8B + LoRA | `evidence_collection_train.jsonl` | ~$4.50 |
| 3 | Controles A.5 (Organizacionales) | Nova Pro | Ministral 3 8B + LoRA | `controls_a5_train.jsonl` | ~$4.50 |
| 4 | Controles A.6 (Personas) | Nova Pro | Ministral 3 8B + LoRA | `controls_a6_train.jsonl` | ~$4.50 |
| 5 | Controles A.8 (Tecnológicos) | Nova Pro | Ministral 3 8B + LoRA | `controls_a8_train.jsonl` | ~$4.50 |
| 6 | Consolidación (Orquestador) | GPT-OSS-120B (OpenAI vía Bedrock) | DeepSeek-R1-Qwen3-8B + LoRA | `consolidation_train.jsonl` | ~$4.50 |
| | **Total** | | | | **~$27.00** |

#### Estrategia en dos fases

**Fase 1 (actual)**: Los agentes usan modelos de Bedrock vía API:
- Sub-agentes → `us.amazon.nova-pro-v1:0` (bajo costo, rápido, español nativo)
- Consolidador → `openai.gpt-oss-120b-1:0` (mejor razonamiento)
- Framework: Strands Agents SDK con patrón "Agents as Tools"

**Fase 2 (posterior)**: Fine-tuning para especialización:
- Sub-agentes → Ministral 3 8B (Apache 2.0, 256K contexto) + QLoRA en SageMaker
- Consolidador → DeepSeek-R1-0528-Qwen3-8B (MIT, chain-of-thought) + QLoRA
- Deploy: Amazon Bedrock AgentCore para hosting de modelos fine-tuned

#### Ventajas de esta arquitectura

1. **Alineación con ISO 27001:2022**: Los agentes de controles mapean directamente a los dominios del Anexo A
2. **Separación de responsabilidades**: Cada agente tiene un scope claro y acotado
3. **Paralelización**: Los 3 agentes de controles pueden ejecutarse en paralelo
4. **Escalabilidad**: Se puede agregar un Agente de Controles Físicos (A.7) sin modificar la arquitectura
5. **Trazabilidad**: Cada hallazgo tiene origen claro (qué agente lo generó)
6. **Costo controlado**: Nova Pro para volumen alto, Claude solo para consolidación final

#### Próximos pasos (Avance 4)

- Implementar el sistema multi-agente con Strands Agents SDK (Fase 1)
- Evaluar calidad de hallazgos generados vs reportes de auditoría reales
- Ejecutar fine-tuning en SageMaker (Fase 2) si la calidad base es insuficiente
- Integrar con Bedrock AgentCore para deployment productivo
- Implementar pipeline completo: usuario → planificación → evidencia → evaluación → reporte


## Conclusiones Actividad 3

### Validación del Baseline

El modelo baseline con **S3 Vectors + Amazon Titan Text Embeddings V2** demuestra que:

1. **Es viable como reemplazo de ChromaDB**: La API de S3 Vectors (`put_vectors`, `query_vectors`) ofrece funcionalidad equivalente a ChromaDB (`upsert`, `query`) con las ventajas de un servicio gestionado.

2. **Los embeddings de Titan V2 (1024d) capturan semántica de dominio**: Las consultas en español e inglés sobre ISO 27001 recuperan chunks relevantes con scores consistentemente > 0.5.

3. **El pipeline es reproducible**: Los datos procesados se almacenan en `s3://mna-s3-proyecto-integrador/processed/` como registro del procesamiento.

4. **El costo es proporcional al uso**: Sin mínimos mensuales, ideal para un proyecto académico con consultas esporádicas.

### Comparación de arquitecturas

| Aspecto | Avance 2 (ChromaDB) | Avance 3 (S3 Vectors) |
|---|---|---|
| Vector store | ChromaDB local | S3 Vectors (cloud) |
| Embeddings | MiniLM 384d (local) | Titan V2 1024d (Bedrock) |
| Persistencia | Disco local | S3 (11 nueves durabilidad) |
| Costo fijo | $0 | $0 (pay-per-use) |
| Integración Bedrock KB | Manual | Nativa |
| Producción-ready | No | Sí |

### Siguientes pasos

Con el baseline validado en S3 Vectors, el siguiente avance integrará este vector store con **Amazon Bedrock Knowledge Bases** para obtener un pipeline RAG completamente gestionado con:
- Parsing automático de documentos (Nova Pro)
- Chunking semántico (NLP-based)
- RetrieveAndGenerate para respuestas fundamentadas
